# 🛡️ AE-WGAN CICIDS-2017 — Version finale

- **Features** : Table 5 (10 features ANOVA)
- **Architecture** : GitHub (BiGAN, activation **linear**, reshape 2×5×1)
- **Dataset équilibré** : downsample BENIGN + augmenter minorités
- **Classifieurs** : tanh (Table 4 CICIDS)
- **Évaluation** : binaire, multiclass, et per-category

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q tensorflow numpy pandas scikit-learn matplotlib scipy

In [ ]:
import os, time
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import (Input, Dense, Reshape, Flatten, Concatenate)
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras import backend as K
from tensorflow.keras import layers, regularizers
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
SEED=42; np.random.seed(SEED); tf.random.set_seed(SEED)
print(f"TF: {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")

## 1. Chargement CICIDS-2017

In [ ]:
# ⚠️ MODIFIER CE CHEMIN SI NÉCESSAIRE
CICIDS_DIR = '/content/drive/MyDrive/cicids2017/MachineLearningCSV/MachineLearningCVE'

if not os.path.exists(CICIDS_DIR):
    print(f"❌ Dossier non trouvé: {CICIDS_DIR}")
    print("Placez les CSV dans le bon dossier Google Drive")
else:
    csv_files = sorted([f for f in os.listdir(CICIDS_DIR) if f.endswith('.csv')])
    print(f"✅ {len(csv_files)} fichiers CSV trouvés")
    for f in csv_files:
        print(f"  {f} ({os.path.getsize(os.path.join(CICIDS_DIR,f))/1e6:.1f} MB)")

In [ ]:
print("Chargement...")
dfs = []
for f in csv_files:
    print(f"  {f}...", end=" ")
    try: df = pd.read_csv(os.path.join(CICIDS_DIR,f), encoding='utf-8', low_memory=False)
    except: df = pd.read_csv(os.path.join(CICIDS_DIR,f), encoding='latin-1', low_memory=False)
    df.columns = df.columns.str.strip()
    print(f"{df.shape[0]} rows")
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
df_all.columns = df_all.columns.str.strip()

label_col = 'Label' if 'Label' in df_all.columns else [c for c in df_all.columns if 'label' in c.lower()][0]
df_all[label_col] = df_all[label_col].str.strip()
print(f"\nDataset: {df_all.shape}")
print(df_all[label_col].value_counts())

## 2. Preprocessing — Table 5 CICIDS

In [ ]:
df_all.replace([np.inf, -np.inf], np.nan, inplace=True)
n_before = len(df_all)
df_all.dropna(inplace=True)
print(f"Supprimé {n_before - len(df_all)} lignes NaN/Inf")

PAPER_FEATS = ['Destination Port','Flow Duration','Bwd Packet Length Min',
    'Flow IAT Mean','Flow IAT Max','Fwd IAT Mean',
    'Fwd Packets/s','Bwd Packets/s','Min Packet Length','min_seg_size_forward']

available = df_all.columns.tolist()
matched = []
for feat in PAPER_FEATS:
    if feat in available: matched.append(feat)
    else:
        found = [c for c in available if feat.replace(' ','').lower() == c.replace(' ','').lower()]
        if found: matched.append(found[0])
        else: print(f"  ⚠️ '{feat}' non trouvée")

print(f"Features: {len(matched)}/10")

y_all = df_all[label_col].copy()
X_all = df_all[matched].copy().apply(pd.to_numeric, errors='coerce')
X_all.dropna(inplace=True)
y_all = y_all.loc[X_all.index]

X_tr_raw, X_te_raw, y_tr_all, y_te_all = train_test_split(
    X_all.values, y_all.values, test_size=0.2, random_state=42, stratify=y_all)

scaler = MinMaxScaler()
X_train_all = scaler.fit_transform(X_tr_raw)
X_test_final = scaler.transform(X_te_raw)

y_train_cat = pd.Series(y_tr_all)
y_test_cat = pd.Series(y_te_all)
INPUT_DIM = X_train_all.shape[1]

print(f"\n✅ Train: {X_train_all.shape}, Test: {X_test_final.shape}, DIM={INPUT_DIM}")
print(f"\nDistribution (train):")
print(y_train_cat.value_counts())

## 3. AE-WGAN CICIDS (linear activations, 2×5)

In [ ]:
class AEWGAN_CICIDS:
    def __init__(self):
        self.img_shape=(2,5,1);self.latent_dim=32
        self.flat_dim=int(np.prod(self.img_shape))
        opt=tf.keras.optimizers.Adamax(0.00009)
        self.discriminator=self._bd()
        self.discriminator.compile(loss=self._wl,optimizer=opt,metrics=['accuracy'])
        self.generator=self._bg()
        self.encoder=self._be()
        self.discriminator.trainable=False
        z=Input(shape=(32,));img_=self.generator(z)
        img=Input(shape=(2,5,1));z_=self.encoder(img)
        self.combined=Model([z,img],[self.discriminator([z,img_]),self.discriminator([z_,img])])
        self.combined.compile(loss=[self._wl,self._wl],optimizer=opt)
    def _wl(self,yt,yp): return K.mean(yt*yp)
    def _be(self):
        m=Sequential([Flatten(input_shape=(2,5,1)),Dense(20,activation='linear'),Dense(32)])
        i=Input(shape=(2,5,1));return Model(i,m(i),name='enc')
    def _bg(self):
        m=Sequential([Dense(20,input_dim=32,activation='linear'),Dense(40,activation='linear'),
            Dense(80,activation='linear'),Dense(120,activation='linear'),Dense(160,activation='linear'),
            Dense(self.flat_dim,activation='linear'),Reshape((2,5,1))])
        z=Input(shape=(32,));return Model(z,m(z),name='gen')
    def _bd(self):
        z=Input(shape=(32,));img=Input(shape=(2,5,1))
        d=Concatenate()([z,Flatten()(img)])
        d=Dense(80,activation='linear')(d);d=Dense(40,activation='linear')(d)
        return Model([z,img],Dense(1,activation='sigmoid')(d),name='disc')
    def train(self,X,epochs=101,batch_size=32,verbose=1):
        r=X.reshape(-1,2,5,1);n=r.shape[0]
        if n<batch_size:batch_size=max(n//2,2)
        v=np.ones((batch_size,1));f=np.zeros((batch_size,1))
        for e in range(epochs):
            z=np.random.normal(0,1,(batch_size,32))
            gi=self.generator.predict(z,verbose=0)
            idx=np.random.randint(0,n,batch_size);imgs=r[idx]
            ze=self.encoder.predict(imgs,verbose=0)
            self.discriminator.train_on_batch([ze,imgs],v)
            self.discriminator.train_on_batch([z,gi],f)
            self.combined.train_on_batch([z,imgs],[v,f])
            if verbose and (e+1)%max(1,epochs//5)==0:
                print(f"      Epoch {e+1}/{epochs}")
    def generate_batch(self,n):
        return self.generator.predict(np.random.normal(0,1,(n,32)),verbose=0,batch_size=256).reshape(-1,self.flat_dim)

print("✅ AEWGAN_CICIDS (linear, 2×5)")

## 4. Construction datasets : 10% threshold + Balanced

In [ ]:
N_BENIGN = (y_train_cat == 'BENIGN').sum()
TARGET_10PCT = int(N_BENIGN * 0.10)
all_labels = sorted(y_train_cat.unique())
n_classes = len(all_labels)

# Target pour dataset équilibré
# On utilise la médiane des tailles de classes comme target
class_sizes = {c: (y_train_cat==c).sum() for c in all_labels}
sorted_sizes = sorted(class_sizes.values())
# Use the size that gives ~70K total, or median, whichever makes sense
TARGET_BALANCED = max(sorted_sizes[len(sorted_sizes)//2], 5000)  # At least 5000

print(f"BENIGN: {N_BENIGN}")
print(f"Target 10%: {TARGET_10PCT}")
print(f"Target balanced: {TARGET_BALANCED} per class ({TARGET_BALANCED * n_classes} total)")
print(f"\nClasses: {n_classes}")
print(f"{'Class':<25} {'Count':>8} {'Action 10%':>15} {'Action Balanced':>18}")
print('─'*70)
for c in all_labels:
    n = class_sizes[c]
    a1 = 'skip' if n >= TARGET_10PCT else f'+{TARGET_10PCT-n}' if n >= 10 else 'too few'
    a2 = 'downsample' if n > TARGET_BALANCED else f'+{TARGET_BALANCED-n}' if n >= 10 else 'too few'
    print(f"  {c:<23} {n:>8} {a1:>15} {a2:>18}")

In [ ]:
# ============================================================
# STRATÉGIE 1: AE-WGAN 10% de BENIGN (comme avant)
# ============================================================
print("="*60)
print(f"STRATÉGIE 1: 10% de BENIGN (seuil={TARGET_10PCT})")
print("="*60)

syn1_X=[]; syn1_y=[]
start=time.time()
for label in all_labels:
    if label == 'BENIGN': continue
    mask = y_train_cat == label
    X_c = X_train_all[mask.values]; n = X_c.shape[0]
    if n >= TARGET_10PCT: print(f"  [{label}] ⏩ {n}>={TARGET_10PCT}"); continue
    if n < 10: print(f"  [{label}] ⏭️ {n}<10"); continue
    ng = TARGET_10PCT - n
    print(f"  [{label}] Training ({n}, +{ng})...", end=" ")
    m = AEWGAN_CICIDS()
    m.train(X_c, epochs=101 if n>500 else 200, batch_size=min(32,max(n//2,2)), verbose=0)
    syn1_X.append(m.generate_batch(ng)); syn1_y.extend([label]*ng)
    print("✅")

print(f"--- {(time.time()-start)/60:.1f} min ---")
if syn1_X:
    X_aug1 = np.vstack([X_train_all, np.vstack(syn1_X)])
    y_aug1 = pd.concat([y_train_cat, pd.Series(syn1_y)], ignore_index=True)
else: X_aug1=X_train_all; y_aug1=y_train_cat
print(f"Dataset 10%: {X_aug1.shape[0]} samples")

In [ ]:
# ============================================================
# STRATÉGIE 2: Dataset équilibré (downsample + augment)
# ============================================================
print("\n" + "="*60)
print(f"STRATÉGIE 2: Dataset équilibré ({TARGET_BALANCED}/classe)")
print("="*60)

bal_X=[]; bal_y=[]
start=time.time()
for label in all_labels:
    mask = y_train_cat == label
    X_c = X_train_all[mask.values]; n = X_c.shape[0]
    if n >= TARGET_BALANCED:
        idx = np.random.choice(n, TARGET_BALANCED, replace=False)
        bal_X.append(X_c[idx]); bal_y.extend([label]*TARGET_BALANCED)
        print(f"  [{label}] ⬇️ {n} → {TARGET_BALANCED}")
    elif n >= 10:
        ng = TARGET_BALANCED - n
        print(f"  [{label}] ⬆️ Training ({n}, +{ng})...", end=" ")
        m = AEWGAN_CICIDS()
        m.train(X_c, epochs=101 if n>500 else 300, batch_size=min(32,max(n//2,2)), verbose=0)
        syn = m.generate_batch(ng)
        bal_X.append(np.vstack([X_c, syn])); bal_y.extend([label]*TARGET_BALANCED)
        print("✅")
    else:
        print(f"  [{label}] ⏭️ {n}<10 — skip")

print(f"--- {(time.time()-start)/60:.1f} min ---")
X_balanced = np.vstack(bal_X)
y_balanced = pd.Series(bal_y)
print(f"Dataset équilibré: {X_balanced.shape[0]} samples")
print(y_balanced.value_counts())

## 5. Classifieurs CICIDS (tanh)

In [ ]:
def build_dt(nc=2): return DecisionTreeClassifier(max_depth=3 if nc==2 else 10,random_state=42)
def build_rf(nc=2): return RandomForestClassifier(100,max_depth=9 if nc==2 else 7,random_state=42,n_jobs=-1)
def build_cnn(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='CNN'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.Conv1D(40,1,activation='tanh',padding='same'))
    if nc>2: m.add(layers.Conv1D(40,1,activation='tanh',padding='same'))
    m.add(layers.Flatten());m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m
def build_gru(dim,nc=2):
    out=1 if nc==2 else nc; act='sigmoid' if nc==2 else 'softmax'
    m=Sequential(name='GRU'); m.add(layers.Input(shape=(dim,1)))
    m.add(layers.GRU(40,activation='tanh',return_sequences=True))
    m.add(layers.GRU(40,activation='tanh',return_sequences=True))
    m.add(layers.GRU(40,activation='tanh'))
    m.add(layers.Dense(40,activation='tanh'))
    m.add(layers.Dense(out,activation=act))
    m.compile(optimizer=Adam(1e-3),loss='binary_crossentropy' if nc==2 else 'sparse_categorical_crossentropy',metrics=['accuracy'])
    return m

print("✅ Classifieurs CICIDS (tanh)")

## 6. Évaluation

In [ ]:
def prepare_labels(y_tr,y_te,binary=True):
    le=LabelEncoder()
    if binary: return (y_tr!='BENIGN').astype(int).values,(y_te!='BENIGN').astype(int).values,le
    else:
        le.fit(list(set(y_tr.unique())|set(y_te.unique())))
        return le.transform(y_tr),le.transform(y_te),le

def train_and_eval(name,clf,X_tr,y_tr,X_te,y_te):
    if name in ['CNN','GRU']:
        X3=X_tr.reshape(-1,X_tr.shape[1],1);Xt3=X_te.reshape(-1,X_te.shape[1],1)
        es=EarlyStopping(monitor='val_loss',patience=5,restore_best_weights=True)
        clf.fit(X3,y_tr,epochs=50,batch_size=256,validation_split=0.1,callbacks=[es],verbose=0)
        yp=clf.predict(Xt3,verbose=0)
        y_pred=(yp.flatten()>0.5).astype(int) if yp.shape[-1]==1 else np.argmax(yp,axis=1)
    else: clf.fit(X_tr,y_tr);y_pred=clf.predict(X_te)
    avg='binary' if len(np.unique(y_te))==2 else 'weighted'
    return {'classifier':name,'accuracy':accuracy_score(y_te,y_pred),
            'precision':precision_score(y_te,y_pred,average=avg,zero_division=0),
            'recall':recall_score(y_te,y_pred,average=avg,zero_division=0),
            'f1_score':f1_score(y_te,y_pred,average=avg,zero_division=0)}

def run_eval(X_tr,y_cat,X_te,y_te_cat,dim,binary=True,label=''):
    tag="BINAIRE" if binary else "MULTICLASSE"
    y_tr,y_te,le=prepare_labels(y_cat,y_te_cat,binary)
    nc=2 if binary else len(np.unique(y_tr))
    print(f"  [{label} {tag}] Classes:{nc} Train:{X_tr.shape[0]}")
    builders={'DT':lambda:build_dt(nc),'RF':lambda:build_rf(nc),
              'CNN':lambda:build_cnn(dim,nc),'GRU':lambda:build_gru(dim,nc)}
    results=[]
    for name,builder in builders.items():
        print(f"    {name}...",end=" ")
        r=train_and_eval(name,builder(),X_tr,y_tr,X_te,y_te)
        print(f"Acc={r['accuracy']:.4f} F1={r['f1_score']:.4f}")
        results.append(r)
    return results

print("✅")

In [ ]:
print("="*60)
print("CLASSIFICATION")
print("="*60)

# Baseline
base_bin = run_eval(X_train_all,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,True,'Baseline')
base_multi = run_eval(X_train_all,y_train_cat,X_test_final,y_test_cat,INPUT_DIM,False,'Baseline')

# AE-WGAN 10%
enh_bin = run_eval(X_aug1,y_aug1,X_test_final,y_test_cat,INPUT_DIM,True,'10%')
enh_multi = run_eval(X_aug1,y_aug1,X_test_final,y_test_cat,INPUT_DIM,False,'10%')

# AE-WGAN Balanced
bal_bin = run_eval(X_balanced,y_balanced,X_test_final,y_test_cat,INPUT_DIM,True,'Balanced')
bal_multi = run_eval(X_balanced,y_balanced,X_test_final,y_test_cat,INPUT_DIM,False,'Balanced')

## 7. Résultats

In [ ]:
print("\n" + "="*70)
print("  COMPARAISON CICIDS-2017")
print("="*70)

for task,br,er,blr in [('BINAIRE',base_bin,enh_bin,bal_bin),('MULTICLASSE',base_multi,enh_multi,bal_multi)]:
    print(f"\n  --- {task} ---")
    print(f"  {'Clf':<6} {'Base F1':>9} {'10% F1':>9} {'Δ 10%':>7} {'Bal F1':>9} {'Δ Bal':>7}")
    print(f"  {'─'*50}")
    for b,e,bl in zip(br,er,blr):
        d1=e['f1_score']-b['f1_score']; d2=bl['f1_score']-b['f1_score']
        print(f"  {b['classifier']:<6} {b['f1_score']:>9.4f} {e['f1_score']:>9.4f} {'+'if d1>=0 else ''}{d1:>6.4f} "
              f"{bl['f1_score']:>9.4f} {'+'if d2>=0 else ''}{d2:>6.4f}")

# Visualization
fig,axes=plt.subplots(1,2,figsize=(14,6))
fig.suptitle('CICIDS-2017: Baseline vs 10% vs Balanced',fontsize=14,fontweight='bold')
for idx,(task,br,er,blr) in enumerate([('Binary',base_bin,enh_bin,bal_bin),('Multiclass',base_multi,enh_multi,bal_multi)]):
    ax=axes[idx]
    clfs=[r['classifier'] for r in br]; x=np.arange(len(clfs)); w=0.25
    ax.bar(x-w,[r['f1_score'] for r in br],w,label='Baseline',color='#3498db',alpha=0.8)
    ax.bar(x,[r['f1_score'] for r in er],w,label='10%',color='#2ecc71',alpha=0.8)
    ax.bar(x+w,[r['f1_score'] for r in blr],w,label='Balanced',color='#e74c3c',alpha=0.8)
    ax.set_xlabel('Classifier');ax.set_ylabel('F1');ax.set_title(task)
    ax.set_xticks(x);ax.set_xticklabels(clfs);ax.legend();ax.set_ylim(0.3,1.05);ax.grid(axis='y',alpha=0.3)
plt.tight_layout()
plt.savefig('results_cicids_final.png',dpi=150,bbox_inches='tight')
plt.show()

# CSV + Drive
rows=[]
for r in base_bin: rows.append({**r,'type':'Baseline','task':'Binary'})
for r in enh_bin: rows.append({**r,'type':'10%','task':'Binary'})
for r in bal_bin: rows.append({**r,'type':'Balanced','task':'Binary'})
for r in base_multi: rows.append({**r,'type':'Baseline','task':'Multiclass'})
for r in enh_multi: rows.append({**r,'type':'10%','task':'Multiclass'})
for r in bal_multi: rows.append({**r,'type':'Balanced','task':'Multiclass'})
df_r=pd.DataFrame(rows)
df_r.to_csv('results_cicids_final.csv',index=False)
import shutil
shutil.copy('results_cicids_final.csv','/content/drive/MyDrive/')
shutil.copy('results_cicids_final.png','/content/drive/MyDrive/')
print("\n✅ Sauvegardé sur Drive")
print(df_r.to_string(index=False))